In [0]:
# Asset Bundle Demo Change
dbutils.widgets.text("source_name", "")

In [0]:
source_name = dbutils.widgets.get("source_name")
print(source_name)

In [0]:
## Read Config Table Based on Widget Value
config = (
    spark.table("dev.demo.ingestion_config")
    .filter(f"source_name = '{source_name}'")
    .first()
)

In [0]:
if not source_name:
    raise ValueError(
        "source_name parameter is mandatory. "
        "Pass it from Workflow or set it manually for testing."
    )

if config is None:
    raise ValueError(
        f"No configuration found for source_name='{source_name}' "
        f"in dev.demo.ingestion_config."
    )

In [0]:
print("Config loaded for:", source_name)
print(config)

In [0]:
source_path = config["source_path"]
target_path = config["target_path"]
file_format = config["file_format"]
load_type = config["load_type"]

In [0]:
print("Source Path:", source_path)
print("Target Path:", target_path)
print("File Format:", file_format)
print("Load Type:", load_type)

In [0]:
## Generic Read Logic
df = (
    spark.read
    .format(file_format)
    .option("recursiveFileLookup", "true")
    .load(source_path)
)

In [0]:
# display(df)

In [0]:
## Add Audit Columns
from pyspark.sql.functions import col, current_timestamp

df = (
    df
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

In [0]:
print("Source:", source_path)
print("Target:", target_path)
print("Rows:", df.count())

In [0]:
## Generic Bronze Write
df.write \
    .format("delta") \
    .mode(load_type) \
    .option("mergeSchema", "true") \
    .save(target_path)

In [0]:
print(f"Successfully loaded {source_name}")
print(f"Written to {target_path}")

In [0]:
df = spark.read.format("delta").load("/Volumes/dev/demo/datasets/bronze/orders")
display(df)